In [1]:
import os
import time
import folium
import tomllib
import shapely
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from typing import Union
from dotenv import load_dotenv
from shapely.geometry import Polygon
from shapely import points, contains, prepare

# Setting Config File

As we have seen in previous notebooks, it is possible to make all the checks for the seismic data using similar vectorization techniques (such as polygon, numeric, etc). However, define and work around each one of the possible checks can be annoying and time-consuming. For this reason, we will define a configuration file where we can set all the checks that we want to perform on the seismic data. This way, we can easily modify the checks without having to change the code. We will use a TOML file for this purpose, as it is a simple and human-readable format.

The main idea here is to design a new scheme using a config file (in format TOML) that allows to define the checks that we want to perform on the seismic data. To achieve this, we will define a TOML file and a wrapper function that reads the config file and executes the checks accordingly. This way, we can easily modify the checks without having to change the code.

## Vectorization functions

As we stated in the Vectorization notebook, we can define different vectorization functions that will be used to perform the checks on the seismic data. In general, we have:

In [2]:
# Numeric comparisons
def numeric_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    threshold : float | None= None,
    lower: float | None = None,
    upper: float | None = None,
    dtype = np.float64
) -> np.ndarray:
    """
    Vectorized numeric comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'eq'       -> values == threshold
            'ne'       -> values != threshold
            'between'  -> lower <= values <= upper
            'outside'  -> values < lower or values > upper
            'abs_gt'   -> abs(values) > threshold
            'abs_ge'   -> abs(values) >= threshold
    threshold : float, optional
        Threshold for one-sided and equality comparisons.
    lower, upper : float, optional
        Bounds for range comparisons.
    dtype : numpy dtype
        Target dtype for NumPy conversion.

    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    values = events[column].to_numpy(dtype=dtype, copy=False)

    if mode == 'gt':
        return values > threshold
    elif mode == 'ge':
        return values >= threshold
    elif mode == 'lt':
        return values < threshold
    elif mode == 'le':
        return values <= threshold
    elif mode == 'eq':
        return values == threshold
    elif mode == 'ne':
        return values != threshold
    elif mode == 'between':
        return (values >= lower) & (values <= upper)
    elif mode == 'outside':
        return (values < lower) | (values > upper)
    elif mode == 'abs_gt':
        return np.abs(values) > threshold
    elif mode == 'abs_ge':
        return np.abs(values) >= threshold
    else:
        raise ValueError(f"Unsupported mode: {mode!r}")

In [3]:
# Column vs column comparisons
def column_column_mask(
    events: pd.DataFrame,
    left_col: str,
    mode: str,
    right_col: str,
    offset: float = 0.0,
    factor: float = 1.0,
    dtype=np.float64,
) -> np.ndarray:
    """
    Vectorized column to column comparator for seismic quality checks.
    It follows: events[left_col] <<mode>> factor * events[right_col] + offset

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    left_col : str
        Left column to compare.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'eq'       -> values == threshold
            'ne'       -> values != threshold
    right_col : str
        Right column to compare.
    offset : float, optional
        Offset of the equation, if required. Defaults to zero.
    factor : float, optional
        Multiplier for the right column, if required. Defaults to zero.
    dtype : numpy dtype
        Target dtype for NumPy conversion.
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    left = events[left_col].to_numpy(dtype=dtype, copy=False)
    right = events[right_col].to_numpy(dtype=dtype, copy=False) * factor + offset
    ops = {
        "gt": np.greater,
        "ge": np.greater_equal,
        "lt": np.less,
        "le": np.less_equal,
        "eq": np.equal,
        "ne": np.not_equal,
    }
    return ops[mode](left, right)

In [4]:
# Non-numeric comparisons
def non_numeric_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    values: list[str] | None = None,
) -> np.ndarray:
    """
    Vectorized non-numeric comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'is_null'       -> values that are null/NaN
            'not_null'      -> values that are not null/NaN
            'in'            -> values that are in the provided list
            'not_in'        -> values that are not in the provided list
    values : list[str], optional
        List of values for 'in' or 'not_in' modes.
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    s = events[column]
    if mode == "is_null":
        return s.isna().to_numpy()
    elif mode == "not_null":
        return s.notna().to_numpy()
    elif mode == "in":
        return s.isin(values).to_numpy()
    elif mode == "not_in":
        return (~s.isin(values)).to_numpy()
    else:
        raise ValueError(f"Unsupported category mode: {mode}")

In [5]:
# Polygonal comparison
def build_polygon_mask(
    events: pd.DataFrame,
    lon_col: str,
    lat_col: str,
    polygon: Union[Polygon, shapely.geometry.base.BaseGeometry],
    mode : str = "inside"
) -> np.ndarray:
    """
    Vectorized polygon comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    lon_col : str
        Longitude column to evaluate.
    lat_col : str
        Latitude column to evaluate.
    polygon : shapely.geometry.Polygon or shapely.geometry.base.BaseGeometry
        Shapely polygon to compare.
    mode : str
        Comparison mode:
            'inside'       -> values inside polygon
            'outside'      -> values outside polygon
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    if mode not in ['inside', 'inside']:
        raise ValueError(f"Unsupported polygon mode: {mode}. Accepted modes: 'inside' and 'outside'")
    if not shapely.is_prepared(polygon):
        shapely.prepare(polygon)
    inside = shapely.contains_xy(
        polygon,
        events[lon_col].to_numpy(),
        events[lat_col].to_numpy()
    )
    return inside if mode == "inside" else ~inside

In [6]:
# Temporal comparison
def temporal_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    value: str,
) -> np.ndarray:
    """
    Vectorized temporal comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > Timestamp
            'ge'       -> values >= Timestamp
            'lt'       -> values < Timestamp
            'le'       -> values <= Timestamp
            'eq'       -> values == Timestamp
            'ne'       -> values != Timestamp
    value : str
        Reference timestamp used for the comparison. Any string accepted by
        :class:`pandas.Timestamp` can be supplied, for example
        '2024-01-01T00:00:00Z' or '2024-01-01 00:00:00'.

    Returns
    -------
    numpy.ndarray
        Boolean mask with one entry per row in events. True indicates
        that the row satisfies the requested temporal condition.
    """
    left = pd.to_datetime(events[column], utc=False)
    right = pd.Timestamp(value)

    ops = {
        "gt": np.greater,
        "ge": np.greater_equal,
        "lt": np.less,
        "le": np.less_equal,
        "eq": np.equal,
        "ne": np.not_equal,
    }
    return ops[mode](left.to_numpy(), right.to_datetime64())

In [7]:
# Composed rules
def combine_masks(
        masks: list[np.ndarray],
        logic: str = "and"
) -> np.ndarray:
    """
    Combine multiple boolean masks using a logical operator.

    Parameters
    ----------
    masks : list[np.ndarray]
        List of boolean masks to combine. All masks must have the same shape.
    logic : str
        Combination logic to apply:
            'and' -> logical AND across all masks
            'or'  -> logical OR across all masks
            'xor' -> logical XOR between exactly 2 masks

    Returns
    -------
    np.ndarray
        Boolean mask with one entry per element in the input masks.

    Raises
    ------
    ValueError
        If no masks are provided, if 'xor' is used with anything other than
        exactly 2 masks, or if an unsupported logic value is supplied.
    """
    if not masks:
        raise ValueError("No masks provided")
    if logic == "and":
        return np.logical_and.reduce(masks)
    elif logic == "or":
        return np.logical_or.reduce(masks)
    elif logic == "xor":
        if len(masks) != 2:
            raise ValueError("XOR logic requires exactly 2 masks")
        return np.logical_xor(masks[0], masks[1])
    else:
        raise ValueError(f"Unsupported logic: {logic!r}")

## Querying datasets

Let's start by querying the seismic data from the database. We will use the `pymysql` library to connect to the database and execute a SQL query to retrieve the seismic data. We will also use the `dotenv` library to load the database credentials from a `.env` file, as stated in the previous notebook.

In [3]:
load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))

# Cutover date: SC3 → SC6
SC6_CUTOVER = dt.datetime(2026, 3, 17, 0, 0, 0)

def _build_connection(prefix: str):
    """Create a pymysql connection using .env credentials for a given prefix."""
    return pymysql.connect(
        host=os.getenv(f'SERVER_{prefix}_HOST'),
        port=int(os.getenv(f'SERVER_{prefix}_PORT', 3306)),
        user=os.getenv(f'SERVER_{prefix}_USERNAME'),
        password=os.getenv(f'SERVER_{prefix}_PASSWORD'),
        database=os.getenv(f'SERVER_{prefix}_DATABASE'),
    )


def _query_db(
    prefix: str,
    query: str,
    start_time: dt.datetime,
    end_time: dt.datetime,
    desc: str,
    **kwargs,
) -> pd.DataFrame:
    """
    Execute a time-bounded SQL query against a single database.

    Parameters
    ----------
    prefix : str
        Credential prefix — 'SC3' or 'SC6'.
    query : str
        Base SQL query ending before the BETWEEN clause.
    start_time, end_time : datetime
        Time bounds for the query.
    desc : str
        Label shown in the tqdm progress bar.
    """
    start_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
    end_str   = end_time.strftime("%Y-%m-%d %H:%M:%S")
    full_query = (
        f"{query} '{start_str}' AND '{end_str}' "
        f"ORDER BY Origin.time_value ASC;"
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        conn = _build_connection(prefix)
        try:
            with tqdm(
                total=1,
                desc=desc,
                unit="query",
                leave=False,
                bar_format="{desc}",
            ) as pbar:
                df = pd.read_sql_query(full_query, conn, **kwargs)
                pbar.update(1)
        finally:
            conn.close()

    return df

def _to_naive_utc(t: dt.datetime) -> dt.datetime:
    """
    Normalize a datetime to naive UTC.
    - Timezone-aware → convert to UTC, strip tzinfo
    - Naive → assumed UTC already, returned as-is
    """
    if t.tzinfo is not None:
        return t.astimezone(dt.timezone.utc).replace(tzinfo=None)
    return t

def connect_to_db(
    query: str,
    start_time: dt.datetime = None,
    end_time: dt.datetime = None,
    **kwargs,
) -> pd.DataFrame:
    """
    Query SC3 and/or SC6 databases depending on the requested time range.

    Decision logic:
        - end_time   <= SC6_CUTOVER  → SC3 only
        - start_time >= SC6_CUTOVER  → SC6 only
        - start_time <  SC6_CUTOVER  < end_time → both (with overlap warning)

    Parameters
    ----------
    query : str
        Base SQL query ending before the BETWEEN clause.
    start_time : datetime, optional
        Start of the time range (UTC). If None, runs the query as-is.
    end_time : datetime, optional
        End of the time range (UTC). If None, runs the query as-is.

    Returns
    -------
    pd.DataFrame
        Query results, merged and sorted by time_value when both DBs are hit.
    """
    # --- No time range: run query as-is against SC3 (legacy default) ----------
    if start_time is None or end_time is None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            conn = _build_connection("SC3")
            try:
                with tqdm(
                    total=1,
                    desc="Querying database...",
                    unit="query",
                    leave=False,
                    bar_format="{desc}",
                ) as pbar:
                    df = pd.read_sql_query(query, conn, **kwargs)
                    pbar.update(1)
            finally:
                conn.close()
        return df

    # --- Normalize to naive UTC before any comparison ------------------------
    start_time = _to_naive_utc(start_time)
    end_time   = _to_naive_utc(end_time)

    # --- Determine which databases are needed --------------------------------
    only_sc3 = end_time   <= SC6_CUTOVER
    only_sc6 = start_time >= SC6_CUTOVER
    both     = not only_sc3 and not only_sc6       # straddles the cutover

    if both:
        warnings.warn(
            f"\n[DATABASE WARNING] The requested time range "
            f"({start_time:%Y-%m-%d %H:%M:%S} → {end_time:%Y-%m-%d %H:%M:%S}) "
            f"spans the seiscomp3-seiscomp6 database cutover ({SC6_CUTOVER:%Y-%m-%d %H:%M:%S} UTC). "
            f"Both databases will be queried and results merged.\n",
            UserWarning,
            stacklevel=2,
        )

    # --- SC3 only ------------------------------------------------------------
    if only_sc3:
        return _query_db(
            prefix="SC3",
            query=query,
            start_time=start_time,
            end_time=end_time,
            desc="Querying SC3 database...",
            **kwargs,
        )

    # --- SC6 only ------------------------------------------------------------
    if only_sc6:
        return _query_db(
            prefix="SC6",
            query=query,
            start_time=start_time,
            end_time=end_time,
            desc="Querying SC6 database...",
            **kwargs,
        )

    # --- Both databases (straddles cutover) ----------------------------------
    # SC3: [start_time, SC6_CUTOVER)
    # SC6: [SC6_CUTOVER, end_time]
    df_sc3 = _query_db(
        prefix="SC3",
        query=query,
        start_time=start_time,
        end_time=SC6_CUTOVER,
        desc="Querying SC3 database (1/2)...",
        **kwargs,
    )
    df_sc6 = _query_db(
        prefix="SC6",
        query=query,
        start_time=SC6_CUTOVER,
        end_time=end_time,
        desc="Querying SC6 database (2/2)...",
        **kwargs,
    )

    # Merge and re-sort by time_value
    df = (
        pd.concat([df_sc3, df_sc6], ignore_index=True)
        .sort_values("time_value")
        .reset_index(drop=True)
    )

    return df

In [4]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

clean_sql = sqlparse.format(revision_query, strip_comments=True).strip()

initial_time = dt.datetime(2023, 1, 1, 0, 0, 0)
final_time = dt.datetime.now(dt.UTC)
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)

print(f"Number of rows in the seismic data: {len(event_df3)}")
# Print how events are by event_type
print("Number of events by event_type:")
print(event_df3['event_type'].value_counts())

/tmp/ipykernel_9596/1579909875.py:9: UserWarning: 
[DATABASE WARNING] The requested time range (2023-01-01 00:00:00 → 2026-06-17 09:01:39) spans the seiscomp3-seiscomp6 database cutover (2026-03-17 00:00:00 UTC). Both databases will be queried and results merged.

  event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)


Number of rows in the seismic data: 248725
Number of events by event_type:
event_type
not locatable                  138619
earthquake                      93797
not existing                     5476
explosion                        5277
outside of network interest      4965
volcanic eruption                 506
induced earthquake                  4
other                               1
Name: count, dtype: int64


In [5]:
# Check size in memory of the seismic data
memory_usage = event_df3.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Memory usage of the seismic data: {memory_usage:.2f} MB")

Memory usage of the seismic data: 142.81 MB
